# 02 · Discretization and G — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)


## 1. Refinement and conditioning

Each added patch adds one dip-slip column. Surface observations smooth depth
structure, so normalized adjacent columns become more alike as patch scale
shrinks.

## 2–3. Component and depth comparisons

Strike and dip columns differ in direction and symmetry. Cosine similarity is
a scale-independent way to compare distinguishability.

## 4. East-only projection

For raw interleaved ENU rows, selecting `0::3` constructs an East-only design
matrix. This is exactly what a one-component observation projector does.


In [ ]:
condition = {}
for n_length, n_width in ((2, 1), (4, 2), (6, 3)):
    fault = geodef.Fault.planar(
        lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
        length=24_000.0, width=12_000.0,
        n_length=n_length, n_width=n_width,
    )
    lon, lat = np.meshgrid(np.linspace(-118.2, -117.8, 6), np.linspace(33.9, 34.1, 5))
    G = fault.greens_matrix(lat.ravel(), lon.ravel())
    G_dip = G[:, fault.n_patches:]
    condition[(n_length, n_width)] = np.linalg.cond(G_dip)
east_only = G[0::3]
assert east_only.shape[0] == lon.size
print(condition, "east-only shape:", east_only.shape)


## Interpretation and alternatives

A growing condition number is expected evidence that representation capacity is outrunning data information.
